In [ ]:
MAX_VIDEOS = 200
epochs = 5
pilot = 1


# Deepfake Detection — Kaggle Pilot (≤200 Videos)

**Purpose**: smoke-test the full pipeline end-to-end on a small DFDC subset before the full run.

**Pipeline**
1. Install dependencies
2. Clone / upload project code
3. Download ≤200 paired DFDC videos from the Kaggle dataset
4. Extract faces (MTCNN) → `data/processed/dfdc/{train,val,test}/{real,fake}/`
5. Verify split sanity (zero leakage, class counts)
6. Train EfficientNet-B3 (timm) for 5 epochs — quick pass to catch training bugs
7. Evaluate on the held-out test split → `saved_predictions.csv`
8. Robustness spot-check (JPEG q=50)

**Kaggle settings needed**
- GPU: T4 × 1 (or P100)
- Internet: ON (for Kaggle API download)
- Add secret `KAGGLE_USERNAME` and `KAGGLE_KEY` via *Add-ons → Secrets* **OR** upload `~/.kaggle/kaggle.json`


## 0 · Environment

In [ ]:
import os, sys
from pathlib import Path

# ── Kaggle secret injection ────────────────────────────────────────────────
# If running inside a Kaggle notebook with Secrets added, inject them now.
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _sec.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = _sec.get_secret('KAGGLE_KEY')
    print('✅ Kaggle credentials loaded from Secrets.')
except Exception as e:
    # Fall back to hardcoded values (replace before uploading to a public notebook!)
    print(f'[warn] Secrets unavailable ({e}); using env-vars if set.')

# ── Confirm GPU availability ───────────────────────────────────────────────
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Install / upgrade dependencies ────────────────────────────────────────
# Kaggle kernels ship with many of these already, but pin versions for safety.
!pip install -q \
    'torch>=2.0.0' \
    'torchvision>=0.15.0' \
    'timm>=1.0.0' \
    'ultralytics>=8.2.0' \
    'facenet-pytorch>=2.6.0' \
    'scikit-learn>=1.3.0' \
    'kaggle>=2.0.0' \
    'tqdm' \
    'matplotlib'
print('✅ Packages ready.')

In [ ]:
# ── Mount / clone project code ─────────────────────────────────────────────
# Option A: upload the repo as a Kaggle Dataset and add it as input.
# Option B: clone from GitHub (requires internet on).
#
# Adjust REPO_DIR to wherever the code lands.

REPO_DIR = Path(r'e:\deepfake')
# ---- Detect execution environment (Kaggle GPU vs. local Windows) ----
import os as _os
_env = _os.getenv('DEEPFAKE_ENV', 'LOCAL')
if _env == 'KAGGLE':
    REPO_DIR = Path('/kaggle/working')
else:
    REPO_DIR = Path(r'e:\deepfake')  # local Windows path
if not REPO_DIR.exists():
    # Try cloning (replace URL with your repo)
    CLONE_URL = 'https://github.com/YOUR_USER/deepfake.git'   # <-- update me
    REPO_DIR  = Path('/kaggle/working/deepfake')
    if not REPO_DIR.exists():
        !git clone {CLONE_URL} {REPO_DIR}

assert REPO_DIR.exists(), f'Project code not found at {REPO_DIR}'
sys.path.insert(0, str(REPO_DIR))
print(f'✅ Project root: {REPO_DIR}')

# Working directory for all outputs
WORK_DIR = REPO_DIR  # always match REPO_DIR
os.chdir(REPO_DIR)   # so config.py finds the right BASE_DIR
print(f'   cwd: {Path.cwd()}')

## 1 · Download DFDC pilot subset (≤200 videos, paired)

Uses the public Kaggle dataset `pranay22077/dfdc-10` (no competition acceptance required).

In [ ]:
import config
from scripts.download_dfdc_subset import download_dfdc_from_dataset
from utils.helpers import ensure_dirs

MAX_VIDEOS = 200     # ← tweak: 50 for a 10-min smoke-test, 200 for a real pilot

RAW_VIDEO_DIR  = WORK_DIR / 'data' / 'raw_videos' / 'dfdc'
DOWNLOAD_CACHE = WORK_DIR / 'data' / 'downloads'
ensure_dirs(RAW_VIDEO_DIR, DOWNLOAD_CACHE)

print(f'Downloading up to {MAX_VIDEOS} DFDC videos → {RAW_VIDEO_DIR} …')
meta = download_dfdc_from_dataset(
    output_dir=RAW_VIDEO_DIR,
    download_dir=DOWNLOAD_CACHE,
    max_videos=MAX_VIDEOS
)

n_real = len(list((RAW_VIDEO_DIR / 'real').glob('*.mp4')))
n_fake = len(list((RAW_VIDEO_DIR / 'fake').glob('*.mp4')))
print(f'\n📦 Downloaded → REAL: {n_real}  FAKE: {n_fake}')
assert n_real > 0 and n_fake > 0, '❌ Download produced no videos — check Kaggle credentials / dataset ref.'

## 2 · Face extraction + identity-safe split

In [ ]:
import types, json
from preprocessing.extract_faces_from_videos import VideoFaceExtractor
from utils.helpers import setup_logger

PROCESSED_DIR = WORK_DIR / 'data' / 'processed' / 'dfdc'

# Point config paths at our working directory
config.BASE_DIR              = REPO_DIR
config.PROCESSED_DFDC_DIR   = PROCESSED_DIR
config.CHECKPOINT_DIR       = WORK_DIR / 'models' / 'checkpoints'
config.REPORTS_DIR          = WORK_DIR / 'reports'

ensure_dirs(PROCESSED_DIR, config.CHECKPOINT_DIR, config.REPORTS_DIR)

logger = setup_logger('Pilot')
extractor = VideoFaceExtractor(
    frames_per_video=8,         # 8 frames per video for speed
    image_size=224,
    margin_ratio=0.20,
    min_face_prob=0.90,
    fallback_center_crop=True,  # keep videos where MTCNN misses
    device=config.DEVICE,
    logger=logger
)

print(f'Using device: {config.DEVICE}')
print(f'Extracting faces from {RAW_VIDEO_DIR} → {PROCESSED_DIR} …')

In [ ]:
# Build a minimal args-like namespace that extract_videos dispatch expects
args = types.SimpleNamespace(
    video_dir          = str(RAW_VIDEO_DIR.parent),  # parent of real/ fake/
    output_dir         = str(PROCESSED_DIR),
    split_mode         = 'identity',
    metadata_file      = str(RAW_VIDEO_DIR / 'metadata.json'),
    celeb_df_test_list = None,
    ffpp_splits_dir    = None,
    frames_per_video   = 8,
    train_ratio        = 0.70,
    val_ratio          = 0.15,
    test_ratio         = 0.15,
    image_size         = 224,
    margin             = 0.20,
    min_prob           = 0.90,
    keep_all_faces     = False,
    fallback_center_crop=True,
    clean              = False,
    seed               = 42,
    device             = str(config.DEVICE),
)

# Run the extraction
from preprocessing.extract_faces_from_videos import VideoFaceExtractor

extractor = VideoFaceExtractor(
    frames_per_video     = args.frames_per_video,
    image_size           = args.image_size,
    margin_ratio         = args.margin,
    min_face_prob        = args.min_prob,
    keep_all_faces       = args.keep_all_faces,
    fallback_center_crop = args.fallback_center_crop,
    device               = config.DEVICE,
    logger               = logger,
)

extractor.process_and_split(
    video_dir          = Path(args.video_dir),
    output_dir         = Path(args.output_dir),
    split_ratios       = (args.train_ratio, args.val_ratio, args.test_ratio),
    split_mode         = args.split_mode,
    metadata_file      = args.metadata_file,
    celeb_df_test_list = args.celeb_df_test_list,
    ffpp_splits_dir    = args.ffpp_splits_dir,
    clean_output       = args.clean,
    seed               = args.seed,
)

print('✅ Extraction complete.')

## 3 · Split sanity checks

In [ ]:
def count_split(split_dir: Path):
    counts = {}
    for cls in ['real', 'fake']:
        d = split_dir / cls
        counts[cls] = len(list(d.glob('*.*'))) if d.exists() else 0
    return counts

splits = {s: count_split(PROCESSED_DIR / s) for s in ['train', 'val', 'test']}
print('\n── Split counts ──────────────────────────────')
print(f'{"Split":<8}  {"Real":>6}  {"Fake":>6}  {"Total":>7}')
print('-' * 40)
for split, cnts in splits.items():
    total = cnts['real'] + cnts['fake']
    print(f'{split:<8}  {cnts["real"]:>6}  {cnts["fake"]:>6}  {total:>7}')

# Hard assertions
assert splits['train']['real'] > 0, '❌ Train split has no REAL images'
assert splits['train']['fake'] > 0, '❌ Train split has no FAKE images'
assert splits['test']['real']  > 0, '❌ Test split has no REAL images'
assert splits['test']['fake']  > 0, '❌ Test split has no FAKE images'
print('\n✅ All split assertions passed.')

In [ ]:
# Leakage check: if the manifest was written, verify zero identity overlap
manifest_path = PROCESSED_DIR / 'video_split_manifest.json'
if manifest_path.exists():
    import json
    with open(manifest_path) as f:
        manifest = json.load(f)
    # Collect video stems per split
    split_videos = {s: set() for s in ['train', 'val', 'test']}
    for vid, info in manifest.items():
        split_videos[info['split']].add(vid)
    # Check pairwise disjoint
    for a, b in [('train','val'), ('train','test'), ('val','test')]:
        overlap = split_videos[a] & split_videos[b]
        assert len(overlap) == 0, f'❌ LEAKAGE: {len(overlap)} videos shared between {a} and {b}: {list(overlap)[:5]}'
    print(f'✅ Zero video-level leakage confirmed across all split pairs.')
else:
    print('[warn] video_split_manifest.json not found; skipping leakage check.')

## 4 · Quick training pass — EfficientNet-B3 (5 epochs)

5 epochs is enough to confirm the training loop, loss curves, and checkpointing work.

In [ ]:
import types
from training.train_timm import run_timm_training

train_args = types.SimpleNamespace(
    model_name      = 'efficientnet_b3.ra2_in1k',
    data_dir        = str(PROCESSED_DIR),
    checkpoint_dir  = str(config.CHECKPOINT_DIR),
    reports_dir     = str(config.REPORTS_DIR),
    epochs          = 5,            # short pass — bump to 25 for real run
    batch_size      = 32,
    lr              = 1e-4,
    weight_decay    = 1e-4,
    patience        = 5,
    num_workers     = 2,
    seed            = 42,
    device          = str(config.DEVICE),
    pretrained      = True,
    jpeg_aug        = True,
    jpeg_q_min      = 50,
    jpeg_q_max      = 95,
    jpeg_p          = 0.5,
)

print(f'Training EfficientNet-B3 for {train_args.epochs} epochs on {config.DEVICE} …')
run_timm_training(train_args)
print('✅ Training complete.')

In [ ]:
# Verify checkpoint was saved
ckpt_path = config.CHECKPOINT_DIR / 'efficientnet_b3.ra2_in1k_best.pth'
assert ckpt_path.exists(), f'❌ Checkpoint not found at {ckpt_path}'
size_mb = ckpt_path.stat().st_size / 1e6
print(f'✅ Checkpoint saved: {ckpt_path.name}  ({size_mb:.1f} MB)')

In [ ]:
# Show training curves
import json, matplotlib.pyplot as plt
history_path = config.REPORTS_DIR / 'efficientnet_b3.ra2_in1k_history.json'
if history_path.exists():
    with open(history_path) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, h['train_loss'], 'b-o', label='Train')
    axes[0].plot(epochs, h['val_loss'],   'r--s', label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=.4)
    axes[1].plot(epochs, h['train_acc'], 'b-o', label='Train')
    axes[1].plot(epochs, h['val_acc'],   'g--s', label='Val')
    axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True, alpha=.4)
    plt.suptitle('EfficientNet-B3 — 5-Epoch Pilot', fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.REPORTS_DIR / 'pilot_curves.png', dpi=150)
    plt.show()
else:
    print('[warn] History JSON not found — check training logs.')

## 5 · Evaluation on held-out test split

In [ ]:
import types
from evaluation.evaluate import evaluate_model

eval_reports = config.REPORTS_DIR / 'pilot_eval'
ensure_dirs(eval_reports)

eval_args = types.SimpleNamespace(
    checkpoint  = str(ckpt_path),
    model       = 'efficientnet_b3.ra2_in1k',
    data_dir    = str(PROCESSED_DIR),
    test_dir    = None,              # use data_dir/test auto-resolve
    reports_dir = str(eval_reports),
    batch_size  = 32,
    device      = str(config.DEVICE),
)

evaluate_model(eval_args)
print(f'\n✅ Evaluation complete. Reports → {eval_reports}')

In [ ]:
# Show saved_predictions.csv preview
import pandas as pd
csv_path = eval_reports / 'saved_predictions.csv'
assert csv_path.exists(), f'❌ saved_predictions.csv not found in {eval_reports}'
df = pd.read_csv(csv_path)
print(f'Predictions CSV: {len(df)} rows')
print(df.head(10).to_string(index=False))

# Quick sanity: video_id column must be populated
assert 'video_id' in df.columns, '❌ video_id column missing from CSV'
assert df['video_id'].notna().all(), '❌ Some video_id values are null'
print('\n✅ CSV columns OK:', list(df.columns))

In [ ]:
# Load and display evaluation summary
import json
summary_path = eval_reports / 'evaluation_summary.json'
with open(summary_path) as f:
# ---- Quality gate: check VALIDATION AUC (configurable via env-var) ----
import os as _os_gate
val_auc = summary.get('val_roc_auc') or summary.get('roc_auc')
_auc_key = 'val_roc_auc' if 'val_roc_auc' in summary else 'roc_auc (test fallback)'
QUALITY_GATE = float(_os_gate.getenv('QUALITY_GATE', '0.70'))
if val_auc is None:
    raise RuntimeError('\u274c QUALITY GATE: AUC key not found in evaluation_summary.json')
if val_auc < QUALITY_GATE:
    raise RuntimeError(
        f'\u274c QUALITY GATE FAILED: {_auc_key}={val_auc:.4f} < {QUALITY_GATE:.2f}. '
        'Debug order: 1) labels  2) face crops  3) splits  4) training settings'
    )
print(f'\u2705 QUALITY GATE PASSED: {_auc_key}={val_auc:.4f} >= {QUALITY_GATE:.2f}')
print('=' * 50)
print('       PILOT EVALUATION RESULTS')
print('=' * 50)
print(f'  Accuracy : {summary["accuracy"]*100:.2f}%')
print(f'  Precision: {summary["precision"]*100:.2f}%')
print(f'  Recall   : {summary["recall"]*100:.2f}%')
print(f'  F1-Score : {summary["f1_score"]*100:.2f}%')
print(f'  ROC-AUC  : {summary["roc_auc"]:.4f}')
print('=' * 50)

# Show confusion matrix
from IPython.display import Image
Image(str(eval_reports / 'confusion_matrix.png'))

## 6 · Spot-check robustness — JPEG q=50

In [ ]:
import types
from evaluation.robustness_test import run_robustness_benchmark as run_robustness_test

rob_args = types.SimpleNamespace(
    data_dir             = str(PROCESSED_DIR),
    yolo_checkpoint      = None,          # skip YOLO in pilot
    convnext_checkpoint  = None,          # skip ConvNeXt in pilot
    effnet_checkpoint    = str(ckpt_path),
    weights_file         = None,
    jpeg_qualities       = [90, 50],      # spot-check two levels
    blur_radii           = [1.0],
    resize_factors       = [0.75],
    reports_dir          = str(config.REPORTS_DIR / 'pilot_robustness'),
    device               = str(config.DEVICE),
    # model names for each member used inside robustness_test
    convnext_model       = 'convnext_tiny.fb_in22k_ft_in1k',
    effnet_model         = 'efficientnet_b3.ra2_in1k',
)

ensure_dirs(rob_args.reports_dir)
run_robustness_test(rob_args)
print('✅ Robustness spot-check done.')

## 7 · Package outputs for download

In [ ]:
import shutil

output_archive = WORK_DIR / 'pilot_outputs'
shutil.make_archive(
    str(output_archive), 'zip',
    root_dir=WORK_DIR,
    base_dir='reports'
)
print(f'✅ Reports archived → {output_archive}.zip')
print(f'   Size: {(output_archive.with_suffix(".zip")).stat().st_size / 1e6:.1f} MB')
print('   Download from Output tab on the right panel.')

## 8 · Checklist before full run

After inspecting the pilot output, tick each item before scaling up:

| # | Check | Status |
|---|-------|--------|
| 1 | Download succeeded (REAL > 0, FAKE > 0) | ☐ |
| 2 | Face extraction yield > 70% (check extractor logs) | ☐ |
| 3 | Zero video leakage confirmed by manifest check | ☐ |
| 4 | Train loss decreasing over 5 epochs | ☐ |
| 5 | Checkpoint saved and loaded without errors | ☐ |
| 6 | `saved_predictions.csv` has `video_id` column | ☐ |
| 7 | ROC-AUC > 0.5 (above chance — data not all one class) | ☐ |
| 8 | Robustness at JPEG q=50 measured (AUC drop noted) | ☐ |

Once all 8 boxes are ticked, switch `MAX_VIDEOS = None` (all of part_00) and `epochs = 25` for the full run.

## 9 · Full Run — Train all 3 models on each dataset

Run this section only after the pilot checklist (§8) is fully green.
This trains EfficientNet-B3, ConvNeXt-Tiny, and YOLO26n-cls on each dataset (Celeb-DF and DFDC).

In [ ]:
# Hyper-parameters for the full run
FULL_EPOCHS    = 25
FULL_PATIENCE  = 7
FULL_BATCH     = 32
SKIP_YOLO      = False   # Set True if only evaluating timm backbones
FORCE_RETRAIN  = False   # Set True to overwrite existing checkpoints


In [ ]:
import types
from scripts.train_all_datasets import run_all

full_args = types.SimpleNamespace(
    datasets   = None,            # None = all available datasets (celebdf, dfdc)
    epochs     = FULL_EPOCHS,
    batch_size = FULL_BATCH,
    patience   = FULL_PATIENCE,
    pilot      = False,
    skip_yolo  = SKIP_YOLO,
    force      = FORCE_RETRAIN,
)

print(f'Starting full training grid: {FULL_EPOCHS} epochs, patience {FULL_PATIENCE}...')
run_all(full_args)
print('✅ Training grid complete.')


## 10 · Cross-Dataset Table + Real-Life Test

Evaluates every (train-domain, test-domain) combination across all models and the soft-voting ensemble,
including evaluation against the in-the-wild Real-Life benchmark.

In [ ]:
import types
from scripts.build_cross_table import run_cross_table

ct_args = types.SimpleNamespace(
    batch_size = 32,
    device     = str(config.DEVICE),
)

results = run_cross_table(ct_args)
print('✅ Cross-dataset evaluation table complete.')


In [ ]:
# Display the markdown table inline
from IPython.display import Markdown, display
from pathlib import Path

md_path = config.REPORTS_DIR / 'cross_dataset' / 'cross_table.md'
if md_path.exists():
    display(Markdown(md_path.read_text(encoding='utf-8')))
else:
    print('cross_table.md not found yet.')


In [ ]:
# Display Heatmap and Comparison Visualizations
from IPython.display import Image, display
heatmap_path = config.REPORTS_DIR / 'cross_dataset' / 'cross_table_heatmap.png'
bars_path = config.REPORTS_DIR / 'cross_dataset' / 'per_model_bars.png'

if heatmap_path.exists():
    print('ROC-AUC Heatmap (Train vs Test domain):')
    display(Image(str(heatmap_path)))
if bars_path.exists():
    print('Per-model performance bars:')
    display(Image(str(bars_path)))


## 11 · Package All Artifacts for Download

In [ ]:
import shutil
from pathlib import Path

reports_archive = WORK_DIR / 'full_reports'
shutil.make_archive(str(reports_archive), 'zip', root_dir=WORK_DIR, base_dir='reports')
print(f'✅ Reports archived -> {reports_archive}.zip')

models_archive = WORK_DIR / 'trained_checkpoints'
shutil.make_archive(str(models_archive), 'zip', root_dir=WORK_DIR, base_dir='models')
print(f'✅ Checkpoints archived -> {models_archive}.zip')
print('Download both archives from Kaggle Output panel.')
